# 第3节. RAG评估
构建RAG系统后，必须回答一个问题：这个系统到底好不好？

RAG评估需要从两个维度同时考量：
- 检索质量 — 搜对了吗？Recall、Precision、MRR
- 生成质量 — 回答对了吗？Faithfulness、Correctness、Relevancy

本节我们就来学习如何系统的对一个RAG系统进行标准化评估。

## 1. RAG评估工具概览
| 工具 | 特点 | 适用场景 |
| --- | --- | --- |
| RAGAS | 开源标准，指标最全 | RAG系统日常评估 |
| LangSmith | LangChain官方平台，追踪+评估一体化 | 追踪+评估一体化 |
| DeepEval | 语法简洁，CI/CD友好 | 自动化测试流水线 |
| TruLens | 可视化反馈闭环 | 调试和迭代优化 |
其中 RAGAS 是社区使用最广的RAG评估框架，提供多个核心指标覆盖检索和生成全流程。

###  1.1 评估数据说明
新版 RAGAS 使用 ragas.Dataset（替代旧的 HuggingFace Dataset）。

新版数据集要求以下4个字段：
| 字段名 | 含义 | 来源 | 需要 Ground Truth 的指标 |
| --- | --- | --- | --- |
| user_input | 用户问题 | 人工或自动生成 | Context Recall / Entities Recall / Noise Sensitivity |
| retrieved_contexts | RAG 系统召回的文档列表 | RAG 检索阶段输出 | 所有检索指标 |
| response | RAG 系统生成的回答 | RAG 生成阶段输出 | Faithfulness / Answer Relevancy / Noise Sensitivity |
| reference | 标准答案 | 人工标注 | Context Recall / Entities Recall / Noise Sensitivity |

数据集示例（ragas.Dataset 单行）：

```json
{
    "user_input": "孔子的教育思想有哪些？",
    "retrieved_contexts": [
        "孔子主张有教无类，认为人人都应该接受教育...",
        "孔子提出因材施教、启发诱导等教学原则..."
    ],
    "response": "孔子主张有教无类，提出因材施教、启发诱导、学思结合等教学原则。",
    "reference": "孔子主张有教无类，提出因材施教、启发诱导、学思结合、谦虚笃实四大教学原则。教育目的是培养士和君子。"
}
```
旧 API 字段名对照：
- question → user_input
- contexts → retrieved_contexts
- answer → response
- ground_truth → reference
新版本中这些字段名是强制的。

### 1.2 安装RAGAS
接下来我们安装RAGAS：
uv add ragas

注意安装的版本，ragas最新是0.4.3版本，如果安装的版本不对可以在pyproject.toml中修改版本号。

新版RAGAS与旧版差异很大，数据集、API都有非常大的变化。

- 兼容性注意：RAGAS 0.4.x 在 ragas/llms/base.py 中硬导入了 langchain_community.chat_models.vertexai，但 langchain-community 0.4+ 已移除该模块。如果遇到 ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'，在 .venv/Lib/site-packages/langchain_community/chat_models/ 下创建 vertexai.py 空壳即可：
    ```python
    from langchain_core.language_models import BaseChatModel
    class ChatVertexAI(BaseChatModel):
        pass
    ```
  根源：RAGAS 把 ChatVertexAI 写成了顶层 import（而非惰性加载），而它实际上只用于 isinstance 检查，从不实例化。这是 RAGAS 对 LangChain 内部模块结构的过度耦合，期待后续版本修复。

### 1.3 RAGAS 核心指标详解

RAGAS 在"RAG评估"类别下提供8个指标（6个文本 + 2个多模态）。下面逐一说明 6个核心文本指标。

```
用户问题 ──→ 检索阶段 ──────────────→ 生成阶段 ──────────→ 回答
              │                           │
              ├─ Context Precision        ├─ Faithfulness
              ├─ Context Recall           ├─ Response Relevancy
              ├─ Context Entities Recall  └─ Noise Sensitivity
              └─ (检索质量)
```

#### 1.3.1 Context Precision（上下文精度）
测什么：检索返回的文档中，有多少是真正和问题相关的？同时考虑排名——相关文档排得越靠前，分数越高。

公式：

$$\text{Context Precision}@K = \frac{\sum_{i=1}^{K} \bigl( \text{relevant}_i \times \frac{|\{\text{relevant in top }i\}|}{i} \bigr)} {|\{\text{total relevant documents}\}|} $$

简化理解：前 K 个结果中，相关文档排得越靠前贡献越大（排第1位权重1，排第3位权重1/3）。

举例：
- 用户问："孔子的教育思想有哪些？"
- 检索返回：① 孔子：有教无类、因材施教 ✓ ② 孟子：性善论 ✗ ③ 荀子：性恶论 ✗
- 只有①相关且排在第一位 → Context Precision ≈ 1.0（高）
- 若顺序为：① 孟子 ✗ ② 荀子 ✗ ③ 孔子 ✓，则相关文档排在第3位 → 分数低

目标：>= 0.85

#### 1.3.2 Context Recall（上下文召回率）
测什么：Ground Truth 中包含的信息，检索回来的文档覆盖了多少？有没有漏掉关键信息？

公式：

$$\text{Context Recall} = \frac{|\text{检索文档能支持的 GT 陈述数}|}{|\text{GT 中总陈述数}|} $$

RAGAS 将 Ground Truth（标准答案） 拆解为原子陈述，逐一判断能否从检索文档中推断出来。

举例：
- 问题："《学记》提出了哪些教学原则？"
- Ground Truth："教学相长、豫时孙摩、长善救失、启发诱导"（4条）
- 检索只返回了"教学相长"和"启发诱导"相关内容 → Context Recall ≈ 2/4 = 0.5（低）
目标：>= 0.70

#### 1.3.3 Context Entities Recall（事实上下文召回）

测什么：Ground Truth 中出现的实体（人名、地名、术语、数字等），检索回来的文档覆盖了多少？适用于事实密集型场景。

公式：

$$\text{Context Entities Recall} = \frac{|\text{RE} \cap \text{RCE}|}{|\text{RE}|} $$

其中 RE = Ground Truth 中的实体集合，RCE = 检索文档中的实体集合。

举例：
- 问题："夸美纽斯有什么贡献？"
- Ground Truth 实体：{夸美纽斯, 大教学论, 班级授课制, 泛智教育, 学年制}
- 检索文档实体：{夸美纽斯, 大教学论, 班级授课制} → 命中3个
- Context Entities Recall = 3/5 = 0.6
目标：>= 0.70

#### 1.3.4 Faithfulness（忠实度 / 反幻觉）

测什么：模型回答中每一个事实陈述是否都能从检索文档中找到依据？检测"幻觉"的核心指标。

公式：

$$\text{Faithfulness} = \frac{|\text{检索文档能支持的陈述数}|}{|\text{回答中的总陈述数}|} $$

RAGAS 将回答拆解为原子级事实陈述，逐一用检索文档验证。

举例：
- 检索文档："夸美纽斯在《大教学论》中提出班级授课制。"
- 模型回答："夸美纽斯提出了班级授课制和泛智教育，被称为教育学之父。"
- 拆解：①"班级授课制" ✓ ②"泛智教育" ✗（文档无） ③"教育学之父" ✗（文档无）
- Faithfulness = 1/3 ≈ 0.33 —— 编造了2个事实
目标：>= 0.90

#### 1.3.5 Response Relevancy（回答相关性 / 切题度）

测什么：模型的回答是否紧扣用户问题？有没有答非所问或跑题？

注：Python 类名为 AnswerRelevancy，文档中称为 Response Relevancy。

公式：

$$\text{Response Relevancy} = \frac{1}{N} \sum_{i=1}^{N} \cos(E_{\text{original\_question}},\  E_{\text{generated\_question}_i}) $$

RAGAS 从模型回答反向生成 N 个问题，计算每个生成问题与原始问题的余弦相似度后取平均。

举例：
- 用户问："教育的本质属性是什么？"
- 回答A："教育是有目的地培养人的社会活动。" → 反向问题 = "教育的本质属性是什么？" → 相似度高
- 回答B："教育经历了原始、古代、现代三个阶段……" → 反向问题 = "教育经历了哪些阶段？" → 相似度低
目标：>= 0.88

#### 1.3.6 Noise Sensitivity（噪声敏感度）

测什么：系统在利用检索文档时，产生错误回答的频率有多高？噪声敏感度越低，系统越鲁棒。

公式：

$$\text{Noise Sensitivity}_{\text{relevant}} = \frac{|\text{回答中的错误陈述数}|}{|\text{回答的总陈述数}|} $$

其中"错误陈述" = 不符合 Ground Truth 或无法归因于相关上下文的陈述。得分越低越好（0 = 完美）。

举例：
- 问题："LIC 以什么著称？"
- Ground Truth："LIC 是印度最大的保险公司，1956年成立，管理大规模投资组合。"
- 模型回答："LIC是最佳保险公司，拥有庞大投资组合，对金融稳定有贡献。"
- 拆解：①"最大保险公司" ✓ ②"投资组合" ✓ ③"对金融稳定有贡献" ✗（Ground Truth 未提及）
- Noise Sensitivity = 1/3 ≈ 0.333
目标：<= 0.10（越低越好）